In [16]:
import os
import re
import random
from collections import Counter
from typing import List, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [17]:
EMBED_DIM = 256
HIDDEN_DIM = 1024
NUM_LAYERS = 1
LR = 0.001
BATCH_SIZE = 64
EPOCHS = 10
GRAD_CLIP = 1.0

MAX_DECODING_LEN = 50
TRAIN_VAL_SPLIT = 0.9

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
SOS_TOKEN = "<sos>"
EOS_TOKEN = "<eos>"

In [18]:
data_file = "rus.txt"
pairs: List[Tuple[str, str]] = []

if not os.path.exists(data_file):
    raise FileNotFoundError(f"{data_file} not found in current folder. Put your rus.txt here.")

with open(data_file, "r", encoding="utf-8") as f:
    for ln in f:
        ln = ln.strip()
        if not ln:
            continue
        parts = ln.split("\t")
        if len(parts) < 2:
            continue
        en, ru = parts[0].strip(), parts[1].strip()
        if en and ru:
            pairs.append((en, ru))

print("Loaded pairs:", len(pairs))
print("Example:", pairs[0])

Loaded pairs: 363386
Example: ('Go.', 'Марш!')


In [19]:
def clean_text(s: str) -> str:
    s = s.lower()
    # remove punctuation (keep letters and whitespace)
    s = re.sub(r"[^\w\s]", " ", s, flags=re.UNICODE)
    s = re.sub(r"\d+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize(s: str) -> List[str]:
    s = clean_text(s)
    if s == "":
        return []
    return s.split()

print(tokenize("Hello, world! 123"))
print(tokenize("Привет, как дела?"))

['hello', 'world']
['привет', 'как', 'дела']


In [20]:
processed_pairs: List[Tuple[List[str], List[str]]] = []
for en, ru in pairs:
    en_toks = tokenize(en)
    ru_toks = tokenize(ru)
    if len(en_toks) == 0 or len(ru_toks) == 0:
        continue
    ru_toks = [SOS_TOKEN] + ru_toks + [EOS_TOKEN]
    processed_pairs.append((en_toks, ru_toks))

print("Processed pairs:", len(processed_pairs))
print("Example:", processed_pairs[0])

Processed pairs: 363386
Example: (['go'], ['<sos>', 'марш', '<eos>'])


In [21]:
def build_vocab(token_lists, min_freq=1, max_size=None, add_special: List[str]=None):
    counter = Counter()
    for toks in token_lists:
        counter.update(toks)
    tokens = [tok for tok, cnt in counter.items() if cnt >= min_freq]
    tokens.sort(key=lambda t: (-counter[t], t))
    if max_size:
        tokens = tokens[:max_size]
    itos = []
    if add_special:
        itos.extend(add_special)
    itos.extend(tokens)
    stoi = {tok: idx for idx, tok in enumerate(itos)}
    return stoi, itos


src_token_lists = [en for en, ru in processed_pairs]
src_stoi, src_itos = build_vocab(src_token_lists, add_special=[PAD_TOKEN, UNK_TOKEN])

tgt_token_lists = [ru for en, ru in processed_pairs]
tgt_stoi, tgt_itos = build_vocab(tgt_token_lists, add_special=[PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN])

PAD_IDX_SRC = src_stoi[PAD_TOKEN]
PAD_IDX_TGT = tgt_stoi[PAD_TOKEN]
UNK_IDX_SRC = src_stoi[UNK_TOKEN]
UNK_IDX_TGT = tgt_stoi[UNK_TOKEN]
SOS_IDX = tgt_stoi[SOS_TOKEN]
EOS_IDX = tgt_stoi[EOS_TOKEN]

print("SRC vocab size:", len(src_itos))
print("TGT vocab size:", len(tgt_itos))
print("Special idxs -> tgt: PAD", PAD_IDX_TGT, "SOS", SOS_IDX, "EOS", EOS_IDX)

SRC vocab size: 15701
TGT vocab size: 53017
Special idxs -> tgt: PAD 0 SOS 5 EOS 4


In [22]:
class TranslationDataset(Dataset):
    def __init__(self, pairs: List[Tuple[List[str], List[str]]], src_stoi: dict, tgt_stoi: dict):
        self.pairs = pairs
        self.src_stoi = src_stoi
        self.tgt_stoi = tgt_stoi

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_toks, tgt_toks = self.pairs[idx]
        src_idxs = [self.src_stoi.get(t, self.src_stoi[UNK_TOKEN]) for t in src_toks]
        tgt_idxs = [self.tgt_stoi.get(t, self.tgt_stoi[UNK_TOKEN]) for t in tgt_toks]
        return torch.tensor(src_idxs, dtype=torch.long), torch.tensor(tgt_idxs, dtype=torch.long)

def collate_batch(batch):
    src_list = [item[0] for item in batch]
    tgt_list = [item[1] for item in batch]
    src_lens = torch.tensor([len(s) for s in src_list], dtype=torch.long)
    tgt_lens = torch.tensor([len(t) for t in tgt_list], dtype=torch.long)
    src_padded = pad_sequence(src_list, batch_first=True, padding_value=src_stoi[PAD_TOKEN])
    tgt_padded = pad_sequence(tgt_list, batch_first=True, padding_value=tgt_stoi[PAD_TOKEN])
    return src_padded, src_lens, tgt_padded, tgt_lens

In [23]:
random.shuffle(processed_pairs)
num_total = len(processed_pairs)
split_idx = int(TRAIN_VAL_SPLIT * num_total)
train_pairs = processed_pairs[:split_idx]
val_pairs = processed_pairs[split_idx:]

train_ds = TranslationDataset(train_pairs, src_stoi, tgt_stoi)
val_ds = TranslationDataset(val_pairs, src_stoi, tgt_stoi)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

print("Train samples:", len(train_ds), "Val samples:", len(val_ds))

Train samples: 327047 Val samples: 36339


In [24]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers=1, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(emb_dim, hid_dim, num_layers=n_layers, batch_first=True)

    def forward(self, src, src_len=None):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)
        return hidden, cell

In [25]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers=1, pad_idx=0):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(emb_dim, hid_dim, num_layers=n_layers, batch_first=True)
        self.fc_out = nn.Linear(hid_dim, output_dim)

    def forward(self, input_token, hidden, cell):
        input_token = input_token.unsqueeze(1)
        embedded = self.embedding(input_token)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell

In [26]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder: Encoder, decoder: Decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, src_len, tgt=None, teacher_forcing_ratio=0.5, max_len=MAX_DECODING_LEN):
        batch_size = src.size(0)
        if tgt is not None:
            tgt_len = tgt.size(1)
            T = tgt_len - 1
        else:
            T = max_len

        outputs = torch.zeros(batch_size, T, self.decoder.output_dim).to(self.device)

        # encode
        hidden, cell = self.encoder(src, src_len)


        if tgt is not None:
            input_tok = tgt[:, 0]
            tgt_out = tgt[:, 1:]
        else:
            input_tok = torch.tensor([SOS_IDX] * batch_size, dtype=torch.long, device=self.device)
            tgt_out = None

        for t in range(T):
            logits, hidden, cell = self.decoder(input_tok, hidden, cell)  # logits: (batch, vocab)
            outputs[:, t, :] = logits

            if tgt is not None:
                # teacher forcing decision
                teacher_force = random.random() < teacher_forcing_ratio
                if teacher_force:
                    input_tok = tgt_out[:, t]
                else:
                    input_tok = logits.argmax(1).detach()
            else:
                input_tok = logits.argmax(1).detach()

        return outputs

In [27]:
INPUT_DIM = len(src_itos)
OUTPUT_DIM = len(tgt_itos)

encoder = Encoder(INPUT_DIM, EMBED_DIM, HIDDEN_DIM, n_layers=NUM_LAYERS, pad_idx=src_stoi[PAD_TOKEN]).to(device)
decoder = Decoder(OUTPUT_DIM, EMBED_DIM, HIDDEN_DIM, n_layers=NUM_LAYERS, pad_idx=tgt_stoi[PAD_TOKEN]).to(device)
model = Seq2Seq(encoder, decoder, device).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=tgt_stoi[PAD_TOKEN])

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Model created. Total params:", total_params)

Model created. Total params: 82436377


In [28]:
def train_epoch(model, loader, optimizer, criterion, teacher_forcing_ratio=0.5):
    model.train()
    epoch_loss = 0.0
    for src_padded, src_lens, tgt_padded, tgt_lens in loader:
        src_padded = src_padded.to(device)
        tgt_padded = tgt_padded.to(device)
        src_lens = src_lens.to(device)

        optimizer.zero_grad()

        outputs = model(src_padded, src_lens, tgt=tgt_padded, teacher_forcing_ratio=teacher_forcing_ratio)


        batch_size, T, vocab_size = outputs.shape
        outputs_flat = outputs.reshape(-1, vocab_size)
        targets_flat = tgt_padded[:, 1:].reshape(-1)

        loss = criterion(outputs_flat, targets_flat)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        epoch_loss += loss.item() * src_padded.size(0)

    return epoch_loss / len(loader.dataset)


def evaluate(model, loader, criterion):
    model.eval()
    epoch_loss = 0.0
    with torch.no_grad():
        for src_padded, src_lens, tgt_padded, tgt_lens in loader:
            src_padded = src_padded.to(device)
            tgt_padded = tgt_padded.to(device)
            src_lens = src_lens.to(device)

            outputs = model(src_padded, src_lens, tgt=tgt_padded, teacher_forcing_ratio=0.0)
            batch_size, T, vocab_size = outputs.shape
            outputs_flat = outputs.reshape(-1, vocab_size)
            targets_flat = tgt_padded[:, 1:].reshape(-1)

            loss = criterion(outputs_flat, targets_flat)
            epoch_loss += loss.item() * src_padded.size(0)

    return epoch_loss / len(loader.dataset)

In [29]:
best_val_loss = float("inf")
save_path = "best_seq2seq.pt"

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, teacher_forcing_ratio=0.5)
    val_loss = evaluate(model, val_loader, criterion)

    print(f"Epoch {epoch}/{EPOCHS}  Train Loss: {train_loss:.4f}  Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            "model_state_dict": model.state_dict(),
            "src_itos": src_itos,
            "src_stoi": src_stoi,
            "tgt_itos": tgt_itos,
            "tgt_stoi": tgt_stoi,
            "config": {
                "EMBED_DIM": EMBED_DIM,
                "HIDDEN_DIM": HIDDEN_DIM,
                "NUM_LAYERS": NUM_LAYERS
            }
        }, save_path)
        print("Saved best model ->", save_path)

Epoch 1/10  Train Loss: 3.5261  Val Loss: 2.8422
Saved best model -> best_seq2seq.pt


KeyboardInterrupt: 

In [30]:
def load_best_model(path="best_seq2seq.pt"):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    print("Loaded model from", path)
    return model

if os.path.exists(save_path):
    load_best_model(save_path)

Loaded model from best_seq2seq.pt


In [31]:
def translate_sentence(model, sentence: str, src_stoi, tgt_itos, max_len=MAX_DECODING_LEN):
    model.eval()
    tokens = tokenize(sentence)
    if len(tokens) == 0:
        return ""

    src_idxs = [src_stoi.get(t, src_stoi[UNK_TOKEN]) for t in tokens]
    src_tensor = torch.tensor(src_idxs, dtype=torch.long).unsqueeze(0).to(device)  # (1, src_len)
    src_len = torch.tensor([len(src_idxs)], dtype=torch.long).to(device)

    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor, src_len)
        input_tok = torch.tensor([SOS_IDX], dtype=torch.long, device=device)
        translated_tokens = []
        for _ in range(max_len):
            logits, hidden, cell = model.decoder(input_tok, hidden, cell)  # (1, vocab)
            next_idx = logits.argmax(1).item()
            if next_idx == EOS_IDX:
                break
            translated_tokens.append(tgt_itos[next_idx])
            input_tok = torch.tensor([next_idx], dtype=torch.long, device=device)
    return " ".join(translated_tokens)

def translate_tensor_input(model, src_tensor: torch.Tensor, src_stoi, tgt_itos, max_len=MAX_DECODING_LEN):
    if src_tensor.dim() == 1:
        src_tensor = src_tensor.unsqueeze(0)
    src_len = torch.tensor([src_tensor.size(1)], dtype=torch.long).to(device)
    src_tensor = src_tensor.to(device)
    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor, src_len)
        input_tok = torch.tensor([SOS_IDX], dtype=torch.long, device=device)
        translated_tokens = []
        for _ in range(max_len):
            logits, hidden, cell = model.decoder(input_tok, hidden, cell)
            next_idx = logits.argmax(1).item()
            if next_idx == EOS_IDX:
                break
            translated_tokens.append(tgt_itos[next_idx])
            input_tok = torch.tensor([next_idx], dtype=torch.long, device=device)
    return " ".join(translated_tokens)

In [32]:
test_sentences = [
    "hello",
    "how are you",
    "good morning",
    "good night",
    "i love you",
    "thank you",
    "what is your name",
    "see you later"
]


for s in test_sentences:
    translation = translate_sentence(model, s, src_stoi, tgt_itos)
    print(f"EN: {s}")
    print(f"RU: {translation}")
    print("-"*40)

EN: hello
RU: птицы
----------------------------------------
EN: how are you
RU: как ты
----------------------------------------
EN: good morning
RU: сегодня утром
----------------------------------------
EN: good night
RU: хорошо
----------------------------------------
EN: i love you
RU: я люблю вас
----------------------------------------
EN: thank you
RU: ты спасибо
----------------------------------------
EN: what is your name
RU: как зовут
----------------------------------------
EN: see you later
RU: ты потом
----------------------------------------
